<a href="https://colab.research.google.com/github/Yesid-Valencia/Deep-Learning---Especializacion-IA/blob/main/R2_A5_S14_Generaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

# Tutorial de DCGAN

**Autor**: [Nathan Inkawhich](https://github.com/inkawhich)

# Introducción

Este tutorial te presentará las Redes Adversarias Generativas Convolucionales Profundas (DCGAN) a través de un ejemplo práctico. Entrenaremos una red adversarial generativa (GAN) para generar nuevas imágenes de rostros de celebridades tras mostrarle fotos de muchas celebridades reales. La mayor parte del código aquí proviene de la implementación de DCGAN en [pytorch/examples](https://github.com/pytorch/examples). Este documento ofrecerá una explicación detallada de la implementación y aclarará cómo y por qué funciona este modelo. No te preocupes, no se requieren conocimientos previos sobre GANs, aunque si es tu primera vez, puede que necesites dedicar algo de tiempo a reflexionar sobre lo que ocurre internamente. Además, para ahorrar tiempo, será de gran ayuda contar con una GPU, o incluso dos. Comencemos desde el principio.

# Redes Adversarias Generativas (GANs)

## ¿Qué es una GAN?

Las GANs son un marco de trabajo para enseñar a un modelo de aprendizaje profundo a capturar la distribución de los datos de entrenamiento, de modo que podamos generar nuevos datos a partir de esa misma distribución. Fueron inventadas por Ian Goodfellow en 2014 y descritas por primera vez en el artículo [Generative Adversarial Nets](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf). Están compuestas por dos modelos distintos: un *generador* y un *discriminador*. El trabajo del generador consiste en crear imágenes "falsas" que se parezcan a las imágenes de entrenamiento. El trabajo del discriminador es analizar una imagen y determinar si es una imagen real del entrenamiento o una imagen falsa generada por el generador. Durante el entrenamiento, el generador intenta constantemente engañar al discriminador generando mejores falsificaciones, mientras que el discriminador se esfuerza por convertirse en un mejor detective para clasificar correctamente las imágenes reales y las falsas. El equilibrio en este juego se alcanza cuando el generador produce imágenes falsas perfectas que parecen extraídas directamente de los datos de entrenamiento, dejando al discriminador con un nivel de confianza de solo el 50% al adivinar si la salida es real o falsa.

Ahora, definamos la notación que utilizaremos a lo largo del tutorial, comenzando con el discriminador. Sea $x$ un dato que representa una imagen. $D(x)$ es la red del discriminador, la cual devuelve la probabilidad (un valor escalar) de que $x$ provenga de los datos de entrenamiento reales en lugar del generador. Como aquí trabajamos con imágenes, la entrada de $D(x)$ es una imagen con dimensiones CHW de 3x64x64. Intuitivamente, $D(x)$ debería ser ALTO cuando $x$ proviene de los datos reales y BAJO cuando proviene del generador. $D(x)$ también puede entenderse como un clasificador binario tradicional.

Para el generador, representamos con $z$ un vector del espacio latente extraído de una distribución normal estándar. $G(z)$ representa la función del generador que mapea el vector latente $z$ al espacio de datos. El objetivo de $G$ es estimar la distribución de la que provienen los datos de entrenamiento ($p_{data}$) para poder generar muestras falsas basadas en esa distribución estimada ($p_g$).

Por lo tanto, $D(G(z))$ es la probabilidad (escalar) de que la salida del generador $G$ sea una imagen real. Como se describe en el [artículo de Goodfellow](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf), $D$ y $G$ juegan un juego minimax en el que $D$ intenta maximizar la probabilidad de clasificar correctamente los datos reales y falsos ($logD(x)$), mientras que $G$ intenta minimizar la probabilidad de que $D$ detecte que sus salidas son falsas ($log(1-D(G(z)))$). Según el artículo, la función de pérdida de la GAN es:

$$\underset{G}{\text{min}} \underset{D}{\text{max}}V(D,G) = \mathbb{E}_{x\sim p_{data}(x)}\big[logD(x)\big] + \mathbb{E}_{z\sim p_{z}(z)}\big[log(1-D(G(z)))\big]$$

En teoría, la solución a este juego minimax se da cuando $p_g = p_{data}$, y el discriminador decide al azar si las entradas son reales o falsas. Sin embargo, la teoría de convergencia de las GANs sigue siendo un área de investigación muy activa y, en la práctica, los modelos no siempre logran entrenarse hasta este punto.

## ¿Qué es una DCGAN?

Una DCGAN es una extensión directa de la GAN descrita anteriormente, con la particularidad de que utiliza explícitamente capas convolucionales y convolucionales transpuestas en el discriminador y en el generador, respectivamente. Fue propuesta por primera vez por Radford et. al. en el artículo [Unsupervised Representation Learning With Deep Convolutional Generative Adversarial Networks](https://arxiv.org/pdf/1511.06434.pdf). El discriminador está compuesto por capas de [convolución con zancada (strided convolutions)](https://pytorch.org/docs/stable/nn.html#torch.nn.Conv2d), capas de [normalización por lotes (batch norm)](https://pytorch.org/docs/stable/nn.html#torch.nn.BatchNorm2d) y activaciones [LeakyReLU](https://pytorch.org/docs/stable/nn.html#torch.nn.LeakyReLU). La entrada es una imagen de 3x64x64 y la salida es una probabilidad escalar de que sea real. El generador consta de capas [convolucionales transpuestas (convolutional-transpose)](https://pytorch.org/docs/stable/nn.html#torch.nn.ConvTranspose2d), capas de normalización por lotes y activaciones [ReLU](https://pytorch.org/docs/stable/nn.html#relu). Su entrada es un vector latente $z$ de una distribución normal estándar y su salida es una imagen RGB de 3x64x64. Estas capas convolucionales transpuestas permiten transformar el vector latente en un volumen con la misma forma que una imagen. En el artículo, los autores también comparten consejos sobre cómo configurar los optimizadores, calcular las pérdidas e inicializar los pesos del modelo, detalles que explicaremos en las siguientes secciones.

In [ ]:
#%matplotlib inline
import argparse
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Configurar semilla aleatoria para reproducibilidad
manualSeed = 999
#manualSeed = random.randint(1, 10000) # Descomenta si quieres nuevos resultados
print("Semilla aleatoria: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Necesario para resultados reproducibles

Semilla aleatoria:  999


# Parámetros de Entrada

Definamos algunos parámetros de entrada para la ejecución:

-   `dataroot`: la ruta a la carpeta raíz del conjunto de datos. Hablaremos más sobre los datos en la próxima sección.
-   `workers`: el número de hilos de trabajo para cargar los datos usando el `DataLoader`.
-   `batch_size`: el tamaño del lote utilizado en el entrenamiento. El artículo de DCGAN utiliza un tamaño de lote de 128.
-   `image_size`: el tamaño espacial de las imágenes de entrenamiento. Esta implementación usa por defecto 64x64. Si se desea usar otro tamaño, se debe cambiar la estructura de D y G. Consulta [aquí](https://github.com/pytorch/examples/issues/70) para más detalles.
-   `nc`: número de canales de color en las imágenes de entrada. Para imágenes a color es 3.
-   `nz`: longitud del vector latente.
-   `ngf`: se relaciona con la profundidad de los mapas de características generados a través del generador.
-   `ndf`: establece la profundidad de los mapas de características propagados a través del discriminador.
-   `num_epochs`: número de épocas de entrenamiento. Entrenar por más tiempo probablemente dará mejores resultados, pero tardará más.
-   `lr`: tasa de aprendizaje para el entrenamiento. Tal como se describe en el artículo de DCGAN, este valor debe ser 0.0002.
-   `beta1`: hiperparámetro beta1 para los optimizadores Adam. Según el artículo, debe ser 0.5.
-   `ngpu`: número de GPUs disponibles. Si es 0, el código se ejecutará en modo CPU. Si es mayor que 0, se ejecutará en esa cantidad de GPUs.

In [ ]:
# Directorio raíz del conjunto de datos
dataroot = "data/celeba"

# Número de hilos de trabajo para el cargador de datos
workers = 2

# Tamaño del lote durante el entrenamiento
batch_size = 128

# Tamaño espacial de las imágenes.
#Se redimensionarán a este tamaño usando un transformador
image_size = 64

# Cantidad de canales de color en las imágenes. Para imágenes a color es 3
nc = 3

# Tamaño del vector latente z (es decir, tamaño de la entrada del generador)
nz = 100

# Tamaño de los mapas de características en el generador
ngf = 64

# Tamaño de los mapas de características en el discriminador
ndf = 64

# Número de épocas de entrenamiento
num_epochs = 5

# Tasa de aprendizaje para los optimizadores
lr = 0.0002

# Hiperparámetro Beta1 para los optimizadores Adam
beta1 = 0.5

# Número de GPUs disponibles. Usa 0 para modo CPU
ngpu = 1

# Datos

En este tutorial utilizaremos el [Celeb-A Faces dataset](http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) que puedes descargar desde el sitio enlazado o en [Google Drive](https://drive.google.com/drive/folders/0B7EVK8r0v71pTUZsaXdaSnZBZzg). El conjunto de datos se descargará como un archivo llamado `img_align_celeba.zip`. Una vez descargado, crea un directorio llamado `celeba` y extrae el archivo zip en él. Luego, configura la variable `dataroot` de este notebook apuntando al directorio `celeba` que acabas de crear. La estructura de directorios resultante debería ser:

``` sh
/ruta/a/celeba
    -> img_align_celeba  
        -> 188242.jpg
        -> 173822.jpg
        -> 284702.jpg
        -> 537394.jpg
           ...
```

Este paso es importante porque utilizaremos la clase de dataset `ImageFolder`, la cual requiere que existan subdirectorios dentro de la carpeta raíz del dataset. Ahora podemos crear el conjunto de datos, el cargador de datos, configurar el dispositivo en el que se ejecutará y, finalmente, visualizar algunas de las imágenes de entrenamiento.

In [ ]:
# Podemos usar un conjunto de datos del tipo ImageFolder con la estructura que tenemos.
# Crear el conjunto de datos
dataset = dset.ImageFolder(root=dataroot,
                           transform=transforms.Compose([
                               transforms.Resize(image_size),
                               transforms.CenterCrop(image_size),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                           ]))
# Crear el cargador de datos
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=workers)

# Decidir en qué dispositivo correremos el código
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

# Graficar algunas imágenes de entrenamiento
real_batch = next(iter(dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Imágenes de Entrenamiento")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/celeba'

# Implementación

Con los parámetros definidos y el conjunto de datos preparado, podemos adentrarnos en la implementación. Comenzaremos con la estrategia de inicialización de pesos y luego explicaremos detalladamente el generador, el discriminador, las funciones de pérdida y el bucle de entrenamiento.

## Inicialización de Pesos

En el artículo de DCGAN, los autores especifican que todos los pesos de los modelos deben inicializarse aleatoriamente a partir de una distribución normal con `media=0` y `desviación_estándar=0.02`. La función `weights_init` toma un modelo inicializado y reconfigura todas sus capas convolucionales, convolucionales transpuestas y de normalización por lotes para cumplir con este criterio. Esta función se aplica a los modelos inmediatamente después de crearlos.

In [ ]:
# Inicialización personalizada de pesos aplicada a ``netG`` y ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Generador

El generador, $G$, está diseñado para mapear el vector del espacio latente ($z$) al espacio de datos. Como nuestros datos son imágenes, convertir $z$ al espacio de datos significa crear una imagen RGB con el mismo tamaño que las imágenes de entrenamiento (es decir, de 3x64x64). En la práctica, esto se logra mediante una serie de capas convolucionales transpuestas bidimensionales con zancada, cada una emparejada con una capa de normalización por lotes 2D y una activación ReLU. La salida del generador pasa por una función Tanh para devolverla al rango de datos de entrada de $[-1,1]$. Cabe destacar la presencia de las capas de normalización por lotes tras las de convolución transpuesta, ya que es una de las contribuciones clave del artículo de DCGAN. Estas capas ayudan a que los gradientes fluyan correctamente durante el entrenamiento. Abajo se muestra un esquema del generador tomado del artículo de DCGAN.

![/\_static/img/dcgan_generator.png](https://pytorch.org/tutorials/_static/img/dcgan_generator.png)

Observa cómo los parámetros que definimos en la sección de entrada (`nz`, `ngf` y `nc`) influyen en la arquitectura del generador en el código. `nz` es la longitud del vector de entrada z, `ngf` se relaciona con el tamaño de los mapas de características que se propagan por el generador, y `nc` es el número de canales de la imagen de salida (establecido en 3 para imágenes RGB). A continuación, se presenta el código del generador.

In [ ]:
# Código del Generador

class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # La entrada es Z, que va directo a una convolución transpuesta
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # Tamaño del estado: ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # Tamaño del estado: ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # Tamaño del estado: ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d( ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # Tamaño del estado: ``(ngf) x 32 x 32``
            nn.ConvTranspose2d( ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # Tamaño del estado final: ``(nc) x 64 x 64``
        )

    def forward(self, input):
        return self.main(input)

Ahora podemos instanciar el generador y aplicar la función `weights_init`. Echa un vistazo al modelo impreso para ver cómo está estructurado el objeto del generador.

In [ ]:
# Crear el generador
netG = Generator(ngpu).to(device)

# Configurar soporte para múltiples GPUs si es necesario
if (device.type == 'cuda') and (ngpu > 1):
    netG = nn.DataParallel(netG, list(range(ngpu)))

# Aplicar la función ``weights_init`` para inicializar de forma aleatoria todos los pesos
# con ``media=0`` y ``desviación_estándar=0.02``.
netG.apply(weights_init)

# Imprimir el modelo
print(netG)

# Discriminador

Como mencionamos, el discriminador, $D$, es una red de clasificación binaria que toma una imagen como entrada y devuelve la probabilidad de que dicha imagen sea real (en lugar de creada de forma artificial). En este caso, $D$ recibe una imagen de entrada de 3x64x64, la procesa mediante una secuencia de capas Conv2d, BatchNorm2d y LeakyReLU, y genera la probabilidad final mediante una función de activación Sigmoid.

Esta arquitectura se puede ampliar con más capas si el problema lo requiere, pero el uso de la convolución con zancada, BatchNorm y LeakyReLUs tiene una importancia fundamental. El artículo de DCGAN señala que es una buena práctica utilizar convoluciones con zancada en lugar de capas de agrupación (pooling) para reducir la resolución, ya que esto permite que la red aprenda sus propias funciones de reducción. Asimismo, la normalización por lotes y las funciones leaky relu favorecen un flujo de gradientes saludable, lo cual resulta crítico para el aprendizaje tanto de $G$ como de $D$.

Código del Discriminador

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # Entrada: ``(nc) x 64 x 64``
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Tamaño del estado: ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # Tamaño del estado: ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # Tamaño del estado: ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # Tamaño del estado: ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

Ahora, de la misma forma que con el generador, podemos crear el discriminador, aplicar la función `weights_init` e imprimir la estructura de la red.

In [ ]:
# Crear el Discriminador
netD = Discriminator(ngpu).to(device)

# Configurar soporte para múltiples GPUs si es necesario
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))

# Aplicar la función ``weights_init`` para inicializar aleatoriamente todos los pesos
# de esta manera: ``media=0, desviación_estándar=0.02``.
netD.apply(weights_init)

# Imprimir el modelo
print(netD)

# Funciones de Pérdida y Optimizadores

Con $D$ y $G$ configurados, podemos definir el método por el cual aprenderán mediante las funciones de pérdida y los optimizadores. Utilizaremos la función de pérdida de entropía cruzada binaria ([BCELoss](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html#torch.nn.BCELoss)), la cual se define en PyTorch de la siguiente manera:

$$\ell(x, y) = L = \{l_1,\dots,l_N\}^\top, \quad l_n = - \left[ y_n \cdot \log x_n + (1 - y_n) \cdot \log (1 - x_n) \right]$$

Observa que esta función calcula ambos componentes logarítmicos de la función objetivo (es decir, $log(D(x))$ y $log(1-D(G(z)))$). Podemos elegir qué parte de la ecuación de BCE calcular a través del parámetro de entrada $y$. Esto se hace en el bucle de entrenamiento que veremos en breve, pero es fundamental comprender que podemos elegir qué componente calcular simplemente modificando $y$ (es decir, las etiquetas reales).

En segundo lugar, definimos nuestra etiqueta real como 1 y la etiqueta falsa como 0. Usaremos estas etiquetas al calcular las pérdidas de $D$ y $G$, que es también la convención empleada en el artículo original de GAN. Por último, configuramos dos optimizadores independientes, uno para $D$ y otro para $G$. Según el artículo de DCGAN, ambos son optimizadores Adam con una tasa de aprendizaje de 0.0002 y Beta1 = 0.5. Para dar seguimiento visual al avance del generador, crearemos un lote fijo de vectores latentes tomados de una distribución gaussiana (`fixed_noise`). En el bucle de entrenamiento, pasaremos de manera periódica este lote por $G$ para observar cómo se forman las imágenes a partir del ruido.

In [ ]:
# Inicializar la función de pérdida ``BCELoss``
criterion = nn.BCELoss()

# Crear un lote de vectores latentes fijos que utilizaremos para visualizar
# el progreso del generador
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Establecer la convención para etiquetas reales y falsas durante el entrenamiento
real_label = 1.
fake_label = 0.

# Configurar los optimizadores Adam tanto para G como para D
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# Entrenamiento

Finalmente, una vez definidos todos los componentes del framework de la GAN, podemos proceder al entrenamiento. Ten en cuenta que entrenar GANs es una tarea compleja, ya que una mala configuración de hiperparámetros suele provocar un colapso del modo (mode collapse) sin ofrecer demasiadas pistas sobre qué ha fallado. Aquí seguiremos de cerca el Algoritmo 1 propuesto en el [artículo de Goodfellow](https://papers.nips.cc/paper/5423-generative-adversarial-nets.pdf), aplicando algunas de las mejores prácticas sugeridas en [ganhacks](https://github.com/soumith/ganhacks). En particular, "construiremos lotes diferentes para las imágenes reales y las falsas" y ajustaremos la función objetivo de G para maximizar $log(D(G(z)))$. El proceso de entrenamiento se divide en dos fases principales: la Fase 1 actualiza el Discriminador y la Fase 2 actualiza el Generador.

**Fase 1 - Entrenar el Discriminador**

Recordemos que el objetivo de entrenar el discriminador consiste en maximizar la probabilidad de clasificar correctamente cualquier entrada como real o falsa. En términos del planteamiento de Goodfellow, buscamos "actualizar el discriminador ascendiendo por su gradiente estocástico". En la práctica, queremos maximizar la expresión $log(D(x)) + log(1-D(G(z)))$. Siguiendo la sugerencia de [ganhacks](https://github.com/soumith/ganhacks) de emplear lotes independientes, realizaremos este cálculo en dos pasos. En primer lugar, crearemos un lote de muestras reales del conjunto de entrenamiento, realizaremos una pasada hacia adelante (forward pass) en $D$, calcularemos la pérdida ($log(D(x))$) y obtendremos los gradientes con una pasada hacia atrás (backward pass). En segundo lugar, generaremos un lote de imágenes falsas con el estado actual de nuestro generador, realizaremos una pasada hacia adelante en $D$, calcularemos la pérdida ($log(1-D(G(z)))$) y *acumularemos* los gradientes sumándolos mediante otra pasada hacia atrás. Con los gradientes de ambos lotes acumulados, procedemos a realizar un paso de optimización en el discriminador.

**Fase 2 - Entrenar el Generador**

De acuerdo con el artículo original, nuestro objetivo es entrenar el generador minimizando la expresión $log(1-D(G(z)))$ para lograr mejores resultados artificiales. Sin embargo, Goodfellow demostró que este enfoque no proporciona gradientes suficientes, especialmente en las etapas iniciales del aprendizaje. Como solución, buscamos maximizar la expresión $log(D(G(z)))$. En nuestro código, logramos esto de la siguiente manera: clasificamos la salida del generador de la Fase 1 con el discriminador, calculamos la pérdida de G *utilizando etiquetas reales*, obtenemos los gradientes de G mediante una pasada hacia atrás y, finalmente, actualizamos los parámetros de G realizando un paso con su optimizador. Puede parecer contradictorio usar etiquetas reales para calcular la pérdida, pero esto nos permite aprovechar la parte $log(x)$ de la función `BCELoss` (en lugar de la parte $log(1-x)$), que es justo lo que requerimos.

Por último, realizaremos un seguimiento de métricas y estadísticas y, al final de cada época, pasaremos nuestro lote `fixed_noise` por el generador para visualizar el progreso del entrenamiento de G. Las estadísticas que reportaremos son:

-   **Loss_D**: pérdida del discriminador calculada como la suma de las pérdidas de los lotes reales y falsos ($log(D(x)) + log(1 - D(G(z)))$).
-   **Loss_G**: pérdida del generador calculada como $log(D(G(z)))$.
-   **D(x)**: promedio de la salida del discriminador (para todo el lote) al evaluar las imágenes reales. Debería comenzar cerca de 1 y luego converger teóricamente a 0.5 a medida que G mejore. Piensa en el porqué de este comportamiento.
-   **D(G(z))**: promedio de la salida del discriminador para el lote de imágenes falsas. El primer número representa el estado antes de actualizar D y el segundo representa el estado posterior. Deberían iniciar cerca de 0 y converger a 0.5 conforme G sea mejor. Considera las razones de este comportamiento.

**Nota:** Este paso puede tardar bastante tiempo en completarse, dependiendo del número de épocas configurado y de si has reducido la cantidad de datos del conjunto de entrenamiento.

In [ ]:
# Bucle de Entrenamiento

# Listas para seguir el progreso
img_list = []
G_losses = []
D_losses = []
iters = 0

print("Iniciando el bucle de entrenamiento...")
# Para cada época
for epoch in range(num_epochs):
    # Para cada lote en el cargador de datos
    for i, data in enumerate(dataloader, 0):

        ############################
        # (1) Actualizar red D: maximizar log(D(x)) + log(1 - D(G(z)))
        ###########################
        ## Entrenar con lote de imágenes reales
        netD.zero_grad()
        # Dar formato al lote
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
        # Pasada hacia adelante del lote real a través de D
        output = netD(real_cpu).view(-1)
        # Calcular la pérdida con el lote real
        errD_real = criterion(output, label)
        # Calcular gradientes para D en la pasada hacia atrás
        errD_real.backward()
        D_x = output.mean().item()

        ## Entrenar con lote de imágenes falsas
        # Generar lote de vectores latentes
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        # Generar lote de imágenes falsas con G
        fake = netG(noise)
        label.fill_(fake_label)
        # Clasificar el lote falso con D
        output = netD(fake.detach()).view(-1)
        # Calcular la pérdida de D con el lote falso
        errD_fake = criterion(output, label)
        # Calcular los gradientes para este lote, acumulándolos (sumándolos) con los anteriores
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        # Calcular la pérdida total de D como la suma de los lotes falsos y reales
        errD = errD_real + errD_fake
        # Actualizar D
        optimizerD.step()


        ############################
        # (2) Actualizar red G: maximizar log(D(G(z)))
        ###########################
        netG.zero_grad()
        label.fill_(real_label)  # Las etiquetas falsas se consideran reales para el costo del generador
        # Dado que acabamos de actualizar D, realizamos otra pasada hacia adelante del lote falso en D
        output = netD(fake).view(-1)
        # Calcular la pérdida de G basándose en esta salida
        errG = criterion(output, label)
        # Calcular los gradientes para G
        errG.backward()
        D_G_z2 = output.mean().item()
        # Actualizar G
        optimizerG.step()

        # Mostrar estadísticas del entrenamiento
        if i % 50 == 0:
            print('[%d/%d][%d/%d]	Loss_D: %.4f	Loss_G: %.4f	D(x): %.4f	D(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        # Guardar pérdidas para graficar más adelante
        G_losses.append(errG.item())
        D_losses.append(errD.item())

        # Evaluar el rendimiento del generador guardando la salida de G sobre fixed_noise
        if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

        iters += 1

# Resultados

Finalmente, analicemos nuestros resultados. Revisaremos tres aspectos distintos. En primer lugar, veremos cómo variaron las pérdidas de D y G durante el entrenamiento. En segundo lugar, visualizaremos el resultado de G en el lote `fixed_noise` para cada época. En tercer lugar, examinaremos un lote de datos reales junto a un lote de datos falsos producidos por G.

**Pérdida en relación con las iteraciones de entrenamiento**

A continuación se muestra un gráfico que compara las pérdidas de D y G frente a las iteraciones de entrenamiento.

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Pérdida del Generador y del Discriminador Durante el Entrenamiento")
plt.plot(G_losses,label="G")
plt.plot(D_losses,label="D")
plt.xlabel("Iteraciones")
plt.ylabel("Pérdida")
plt.legend()
plt.show()

**Visualización de la evolución de G**

Como recordarás, guardamos las imágenes de salida del generador para el lote `fixed_noise` tras cada época. Ahora podemos visualizar este progreso del entrenamiento mediante una animación. Haz clic en el botón de reproducción para iniciar el video.

In [ ]:
fig = plt.figure(figsize=(8,8))
plt.axis("off")
ims = [[plt.imshow(np.transpose(i,(1,2,0)), animated=True)] for i in img_list]
ani = animation.ArtistAnimation(fig, ims, interval=1000, repeat_delay=1000, blit=True)

HTML(ani.to_jshtml())

**Imágenes Reales vs. Imágenes Falsas**

Para terminar, observemos y comparemos un lote de imágenes reales frente a uno de imágenes falsas frente a frente.

In [ ]:
# Obtener un lote de imágenes reales del cargador de datos
real_batch = next(iter(dataloader))

# Graficar las imágenes reales
plt.figure(figsize=(15,15))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Imágenes Reales")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=5, normalize=True).cpu(),(1,2,0)))

# Graficar las imágenes falsas de la última época
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Imágenes Falsas")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()